# Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.svm import SVC, LinearSVC
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler

## NLTK Word2Vec

In [4]:
!pip install gensim
from gensim.models import Word2Vec
nltk.download('all')
import nltk
for package in ['stopwords','punkt','wordnet']:
    nltk.download(package) 
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words('english')) 

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     C:\Users\Josen\AppData\Roaming\nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     C:\Users\Josen\AppData\Roaming\nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     C:\Users\Josen\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     C:\Users\Josen\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     C:\Users\Josen\AppData\Roaming\nltk_data...
[

In [5]:
# Download and load the data
import keras
import os

f_path_1 = "data/train.csv.zip"
url_1 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/train.csv.zip"
if not os.path.exists(f_path_1):
    file_1 = keras.utils.get_file(f_path_1, url_1)
train_df = pd.read_csv(f_path_1)

f_path_2 = "data/test.csv.zip"
url_2 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/test.csv.zip"
if not os.path.exists(f_path_2):
    file_2 = keras.utils.get_file(f_path_2, url_2)
test_df = pd.read_csv(f_path_2)

In [6]:
## Tokenzier & Lemmatization
class lemmaTokenizer(object):
    def __init__(self, stop_words):
        self.stop_words = stop_words
        from nltk.stem import WordNetLemmatizer
        self.lemmatizer = WordNetLemmatizer()
    def __call__(self, doc):
        tokens = word_tokenize(doc)
        filtered_tok = []
        for tok in tokens:
            if tok not in stop_words:
                #tok = re.sub('\W+','', tok) #Punctuation strip
                tok = re.sub(r"[^\w']+", '', tok) ## version that allows apostrophe
                tmp = self.lemmatizer.lemmatize(tok)
                if len(tmp) >= 2:
                    filtered_tok.append(tmp)
        return filtered_tok

In [7]:
#for converting sentence to vectors/numbers from word vectors result by Word2Vec
class MeanEmbeddingVectorizer(object):
    def __init__(self, word2vec):
        self.word2vec = word2vec
        # if a text is empty we should return a vector of zeros
        # with the same dimensionality as all the other vectors
        self.dim = len(next(iter(word2vec.values())))

    def fit(self, X, y):
        return self

    def transform(self, X):
        return np.array([
            np.mean([self.word2vec[w] for w in words if w in self.word2vec]
                    or [np.zeros(self.dim)], axis=0)
            for words in X
        ])

# Project 1 - NLP and Text Classification

For this project you will need to classify some angry comments into their respective category of angry. The process that you'll need to follow is (roughly):
<ol>
<li> Use NLP techniques to process the training data. 
<li> Train model(s) to predict which class(es) each comment is in.
    <ul>
    <li> A comment can belong to any number of classes, including none. 
    </ul>
<li> Generate predictions for each of the comments in the test data. 
<li> Write your test data predicitions to a CSV file, which will be scored. 
</ol>

You can use any models and NLP libraries you'd like. 

## Training Data

Use the training data to train your prediction model(s). Each of the classification output columns (toxic to the end) is a human label for the comment_text, assessing if it falls into that category of "rude". A comment may fall into any number of categories, or none at all. Membership in one output category is <b>independent</b> of membership in any of the other classes (think about this when you plan on how to make these predictions - it may also make it easier to split work amongst a team...). 

In [8]:
#train_df = pd.read_csv("train.csv.zip")
train_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


## Test Data

In [9]:
#test_df = pd.read_csv("test.csv")
test_df.head()

,id,comment_text
0,1,Yo bitch Ja Rule is more succesful then you'll...
1,2,== From RfC == \n\n The title is fine as it is...
2,3,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,4,":If you have a look back at the source, the in..."
4,5,I don't anonymously edit articles at all.


In [10]:
## Creating a working copy of test dataframe
df_test = test_df.copy()

## Tokenizing & Lemmatizing/cleaning the text in test data
test_tok = lemmaTokenizer(stopwords)
df_test["clean_text"] = df_test["comment_text"].apply(lambda x: test_tok(x))
df_test.head()

,id,comment_text,clean_text
0,1,Yo bitch Ja Rule is more succesful then you'll...,"[Yo, bitch, Ja, Rule, succesful, 'll, ever, wh..."
1,2,== From RfC == \n\n The title is fine as it is...,"[From, RfC, The, title, fine, IMO]"
2,3,""" \n\n == Sources == \n\n * Zawe Ashton on Lap...","[Sources, Zawe, Ashton, Lapland]"
3,4,":If you have a look back at the source, the in...","[If, look, back, source, information, updated,..."
4,5,I don't anonymously edit articles at all.,"[n't, anonymously, edit, article]"


In [11]:
X_test_v = df_test["clean_text"]

## Vincent - Columns: Obscene, Threat

### Prepping dataframe

In [12]:
df_v = train_df[["id", "comment_text", "obscene", "threat"]].copy()
df_v.head()

,id,comment_text,obscene,threat
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0


In [13]:
tok_v = lemmaTokenizer(stopwords)
df_v["clean_text"] = df_v["comment_text"].apply(lambda x: tok_v(x))
df_v.head()

,id,comment_text,obscene,threat,clean_text
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,"[Explanation, Why, edits, made, username, Hard..."
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,"[D'aww, He, match, background, colour, 'm, see..."
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,"[Hey, man, 'm, really, trying, edit, war, It, ..."
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,"[More, ca, n't, make, real, suggestion, improv..."
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,"[You, sir, hero, Any, chance, remember, page, 's]"


In [14]:
## Arranging columns better
df_v = df_v[["id", "comment_text", "clean_text", "obscene", "threat"]]
df_v.head()

,id,comment_text,clean_text,obscene,threat
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,"[Explanation, Why, edits, made, username, Hard...",0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,"[D'aww, He, match, background, colour, 'm, see...",0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...","[Hey, man, 'm, really, trying, edit, war, It, ...",0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...","[More, ca, n't, make, real, suggestion, improv...",0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...","[You, sir, hero, Any, chance, remember, page, 's]",0,0


In [15]:
X_train_v = df_v["clean_text"]
y_obs = df_v["obscene"]
y_threat = df_v["threat"]

## Josenta Columns - Insult and Identity Hate

In [16]:
train_df.columns

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')

In [17]:
df_j = train_df[["id", "comment_text", "identity_hate", "insult"]].copy()
df_j.head()

,id,comment_text,identity_hate,insult
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0


In [18]:
tok_v = lemmaTokenizer(stopwords)
df_j["clean_text"] = df_j["comment_text"].apply(lambda x: tok_v(x))
df_j.head()

,id,comment_text,identity_hate,insult,clean_text
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,"[Explanation, Why, edits, made, username, Hard..."
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,"[D'aww, He, match, background, colour, 'm, see..."
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,"[Hey, man, 'm, really, trying, edit, war, It, ..."
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,"[More, ca, n't, make, real, suggestion, improv..."
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,"[You, sir, hero, Any, chance, remember, page, 's]"


In [19]:
X_train_j = df_j["clean_text"]
y_idh = df_j["identity_hate"]
y_insult = df_j["insult"]

In [20]:
## Downloading models from gensim to use their pre-trained W2v model
import gensim.downloader as api
model_train = api.load("glove-twitter-50")
#model_train = api.load("glove-twitter-100")
w2v_train = dict(zip(model_train.index_to_key, model_train.vectors))

[==================================================] 100.0% 199.5/199.5MB downloaded


In [21]:
modelW_train = MeanEmbeddingVectorizer(w2v_train)

X_train_vectors_w2v = modelW_train.transform(X_train_j)
X_test_vectors_w2v = modelW_train.transform(X_test_v)

print("Train shape", X_train_vectors_w2v.shape, "Test shape", X_test_vectors_w2v.shape)
pd.DataFrame(X_train_vectors_w2v)

Train shape (159571, 50) Test shape (153164, 50)


,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,0.216505,0.263524,-0.092532,-0.092776,0.196199,-0.514459,0.806201,-0.180284,0.244406,-0.048628,...,-0.091213,0.101148,0.344130,-0.171172,-0.006699,-0.369154,-0.053883,-0.099569,-0.351240,-0.202192
1,0.001638,-0.093660,0.149412,0.134436,0.041293,0.235891,1.097069,-0.114139,0.210120,0.487536,...,-0.338484,0.190001,0.245546,0.066416,0.033574,-0.123512,0.312956,0.014116,0.022955,-0.198975
2,0.366375,0.311255,-0.098029,-0.092966,0.165751,0.107282,0.783589,-0.167035,0.129563,0.030519,...,-0.387555,0.342561,0.221675,0.021111,0.097078,-0.209228,0.221928,0.075896,-0.414054,-0.241107
3,0.296372,0.190718,-0.192479,0.060039,0.184215,-0.026242,0.699315,-0.222459,0.185733,0.221716,...,-0.376051,0.214210,0.291949,0.027453,0.081779,-0.295740,0.024700,-0.031314,-0.248680,-0.038186
4,0.232730,0.082843,0.167632,-0.222937,-0.092281,-0.086910,0.785938,0.235950,-0.190176,-0.063731,...,-0.795572,-0.330162,0.098599,0.312315,-0.068577,-0.105242,0.005909,-0.338990,-0.041987,-0.273363
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159566,0.404827,0.175646,-0.040050,0.376392,0.040849,0.203373,0.730878,-0.179062,-0.029947,0.270713,...,-0.568661,0.233869,0.409587,-0.199086,0.150220,-0.280967,0.055375,0.121509,-0.177176,-0.002076
159567,0.508747,0.226817,-0.019533,0.096389,0.274962,0.005065,1.025753,0.086674,0.030419,0.402622,...,-0.350408,0.204942,0.311302,0.032799,0.098613,-0.179711,0.483027,0.376045,-0.142635,-0.193166
159568,0.167532,0.285695,-0.064891,0.227676,0.567518,-0.139942,0.642592,-0.061124,0.322034,-0.066632,...,-0.375272,-0.284672,0.143458,0.219418,0.373556,-0.126718,0.355566,0.285052,-0.346069,-0.030145
159569,-0.107045,0.225700,0.121029,-0.069840,0.072886,-0.209905,0.997228,0.196204,0.013345,0.354489,...,-0.563216,-0.124924,0.077784,0.043101,0.077829,-0.496654,0.238414,0.066508,-0.425342,0.063336


In [22]:
## Making Predictions
rfc_j = RandomForestClassifier(n_jobs=-1, max_depth=10)
## fitting for obscene target
rfc_j.fit(X_train_vectors_w2v, y_idh)

## Predicting for obscene target
y_preds = rfc_j.predict(X_test_vectors_w2v)

In [25]:
## Fitting for threats target
rfc_j.fit(X_train_vectors_w2v, y_insult)
## Generating predictions for threat target
ythreat_preds = rfc_j.predict(X_test_vectors_w2v)

### Creating Word2vec Model &/or trying pre-trained model

In [26]:
# ## Creating Word2vec model for training based on our corpus
# model_train = Word2Vec(df_v["clean_text"], min_count=3, vector_size=200, sg=1)

# ## word & vector combo
# w2v_train = dict(zip(model_train.wv.index_to_key, model_train.wv.vectors))

In [ ]:
## Downloading models from gensim to use their pre-trained W2v model
import gensim.downloader as api
model_train = api.load("glove-twitter-50")
#model_train = api.load("glove-twitter-100")
w2v_train = dict(zip(model_train.index_to_key, model_train.vectors))

In [27]:
modelW_train = MeanEmbeddingVectorizer(w2v_train)

X_train_vectors_w2v = modelW_train.transform(X_train_v)
X_test_vectors_w2v = modelW_train.transform(X_test_v)

print("Train shape", X_train_vectors_w2v.shape, "Test shape", X_test_vectors_w2v.shape)
pd.DataFrame(X_train_vectors_w2v)

Train shape (159571, 50) Test shape (153164, 50)


,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,0.216505,0.263524,-0.092532,-0.092776,0.196199,-0.514459,0.806201,-0.180284,0.244406,-0.048628,...,-0.091213,0.101148,0.344130,-0.171172,-0.006699,-0.369154,-0.053883,-0.099569,-0.351240,-0.202192
1,0.001638,-0.093660,0.149412,0.134436,0.041293,0.235891,1.097069,-0.114139,0.210120,0.487536,...,-0.338484,0.190001,0.245546,0.066416,0.033574,-0.123512,0.312956,0.014116,0.022955,-0.198975
2,0.366375,0.311255,-0.098029,-0.092966,0.165751,0.107282,0.783589,-0.167035,0.129563,0.030519,...,-0.387555,0.342561,0.221675,0.021111,0.097078,-0.209228,0.221928,0.075896,-0.414054,-0.241107
3,0.296372,0.190718,-0.192479,0.060039,0.184215,-0.026242,0.699315,-0.222459,0.185733,0.221716,...,-0.376051,0.214210,0.291949,0.027453,0.081779,-0.295740,0.024700,-0.031314,-0.248680,-0.038186
4,0.232730,0.082843,0.167632,-0.222937,-0.092281,-0.086910,0.785938,0.235950,-0.190176,-0.063731,...,-0.795572,-0.330162,0.098599,0.312315,-0.068577,-0.105242,0.005909,-0.338990,-0.041987,-0.273363
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159566,0.404827,0.175646,-0.040050,0.376392,0.040849,0.203373,0.730878,-0.179062,-0.029947,0.270713,...,-0.568661,0.233869,0.409587,-0.199086,0.150220,-0.280967,0.055375,0.121509,-0.177176,-0.002076
159567,0.508747,0.226817,-0.019533,0.096389,0.274962,0.005065,1.025753,0.086674,0.030419,0.402622,...,-0.350408,0.204942,0.311302,0.032799,0.098613,-0.179711,0.483027,0.376045,-0.142635,-0.193166
159568,0.167532,0.285695,-0.064891,0.227676,0.567518,-0.139942,0.642592,-0.061124,0.322034,-0.066632,...,-0.375272,-0.284672,0.143458,0.219418,0.373556,-0.126718,0.355566,0.285052,-0.346069,-0.030145
159569,-0.107045,0.225700,0.121029,-0.069840,0.072886,-0.209905,0.997228,0.196204,0.013345,0.354489,...,-0.563216,-0.124924,0.077784,0.043101,0.077829,-0.496654,0.238414,0.066508,-0.425342,0.063336


### Making Predictions & Adding them to test dataframe

In [28]:
## Making Predictions
rfc_w2v = RandomForestClassifier(n_jobs=-1, max_depth=10)
## fitting for obscene target
rfc_w2v.fit(X_train_vectors_w2v, y_obs)

## Predicting for obscene target
yobs_preds = rfc_w2v.predict(X_test_vectors_w2v)

In [29]:
## Fitting for threats target
rfc_w2v.fit(X_train_vectors_w2v, y_threat)
## Generating predictions for threat target
ythreat_preds = rfc_w2v.predict(X_test_vectors_w2v)

In [30]:
## Adding obscene target predictions to test dataframe
df_test["obscene_prediction"] = yobs_preds
## Adding threat target predictions to test dataframe
df_test["threat_prediction"] = ythreat_preds

df_test

,id,comment_text,clean_text,obscene_prediction,threat_prediction
0,1,Yo bitch Ja Rule is more succesful then you'll...,"[Yo, bitch, Ja, Rule, succesful, 'll, ever, wh...",1,0
1,2,== From RfC == \n\n The title is fine as it is...,"[From, RfC, The, title, fine, IMO]",0,0
2,3,""" \n\n == Sources == \n\n * Zawe Ashton on Lap...","[Sources, Zawe, Ashton, Lapland]",0,0
3,4,":If you have a look back at the source, the in...","[If, look, back, source, information, updated,...",0,0
4,5,I don't anonymously edit articles at all.,"[n't, anonymously, edit, article]",0,0
...,...,...,...,...,...
153159,153160,". \n i totally agree, this stuff is nothing bu...","[totally, agree, stuff, nothing, toolongcrap]",0,0
153160,153161,== Throw from out field to home plate. == \n\n...,"[Throw, field, home, plate, Does, get, faster,...",0,0
153161,153162,""" \n\n == Okinotorishima categories == \n\n I ...","[Okinotorishima, category, see, change, agree,...",0,0
153162,153163,""" \n\n == """"One of the founding nations of the...","['', One, founding, nation, EU, Germany, Law, ...",0,0


In [31]:
df_test[df_test["obscene_prediction"] == 1]

,id,comment_text,clean_text,obscene_prediction,threat_prediction
0,1,Yo bitch Ja Rule is more succesful then you'll...,"[Yo, bitch, Ja, Rule, succesful, 'll, ever, wh...",1,0
48,49,DJ Robinson is gay as hell! he sucks his dick ...,"[DJ, Robinson, gay, hell, suck, dick, much]",1,0
59,60,":Fuck off, you anti-semitic cunt. |","[Fuck, antisemitic, cunt]",1,0
70,71,== Hello == \n\n Fuck off my Pagan you barebac...,"[Hello, Fuck, Pagan, bareback, mancunt, pig, s...",1,0
99,100,Stone Sour sucks anus,"[Stone, Sour, suck, anus]",1,0
...,...,...,...,...,...
153083,153084,They shouldnt give shit to this racist ass bas...,"[They, shouldnt, give, shit, racist, as, bastard]",1,0
153086,153087,"i does what i wants i am an eskimo whisperer, ...","[want, eskimo, whisperer, shut, yo, face]",1,0
153090,153091,Dear sir: YOU are a fucking cunt. Rahm Emanue...,"[Dear, sir, YOU, fucking, cunt, Rahm, Emanuel,...",1,0
153106,153107,youshit dick cock fuckshit dick cock fuck shit...,"[youshit, dick, cock, fuckshit, dick, cock, fu...",1,0


## Output Details, Submission Info, and Example Submission

For this project, please output your predictions in a CSV file. The structure of the CSV file should match the structure of the example below. 

The output should contain one row for each row of test data, complete with the columns for ID and each classification.

Into Moodle please submit:
<ul>
<li> Your notebook file(s). I'm not going to run them, just look. 
<li> Your sample submission CSV. This will be evaluated for accuracy against the real labels; only a subset of the predictions will be scored. 
</ul>

It is REALLY, REALLY, REALLY important the the structure of your output matches the specifications. The accuracies will be calculated by a script, and it is expecting a specific format. 

### Sample Evaluator

The file prediction_evaluator.ipynb contains an example scoring function, scoreChecker. This function takes a sumbission and an answer key, loops through, and evaluates the accuracy. You can use this to verify the format of your submission. I'm going to use the same function to evaluate the accuracy of your submission, against the answer key (unless I made some mistake in this counting function).

In [32]:
#Construct dummy data for a sample output. 
#You won't do this part, you have real data
#Your data should have the same structure, so the CSV output is the same
dummy_ids = ["dfasdf234", "asdfgw43r52", "asdgtawe4", "wqtr215432"]
dummy_toxic = [0,0,0,0]
dummy_severe = [0,0,0,0]
dummy_obscene = [0,1,1,0]
dummy_threat = [0,1,0,1]
dummy_insult = [0,0,1,0]
dummy_ident = [0,1,1,0]
columns = ["id", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
sample_out = pd.DataFrame( list(zip(dummy_ids, dummy_toxic, dummy_severe, dummy_obscene, dummy_threat, dummy_insult, dummy_ident)),
                    columns=columns)
sample_out.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,dfasdf234,0,0,0,0,0,0
1,asdfgw43r52,0,0,1,1,0,1
2,asdgtawe4,0,0,1,0,1,1
3,wqtr215432,0,0,0,1,0,0


In [33]:
#Write DF to CSV. Please keep the "out.csv" filename. Moodle will auto-preface it with an identifier when I download it. 
#This command should work with your dataframe of predictions. 
sample_out.to_csv('output/out.csv', index=False)  

## Grading

The grading for this is split between accuracy and well written code:
<ul>
<li> 75% - Accuracy. The most accurate will get 100% on this, the others will be scaled down from there. 
<li> 25% - Code quality. Can the code be followed and made sense of - i.e. comments, sections, titles. 
</ul>